In [1]:
print("Hello, World!")

Hello, World!


In [1]:
import os

In [2]:
os.getenv('OLLAMA_MODEL')

'gpt-oss:120b-cloud'

In [5]:
from langchain.tools import tool
from langchain.chat_models import init_chat_model
from langchain_ollama.chat_models import ChatOllama

model = ChatOllama(
    model = os.getenv('OLLAMA_MODEL'),
    base_url="http://localhost:11434"
)

In [6]:
# Define tools
@tool
def multiply(a: int, b: int) -> int:
    """Multiply `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a * b


@tool
def add(a: int, b: int) -> int:
    """Adds `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a + b


@tool
def divide(a: int, b: int) -> float:
    """Divide `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a / b


In [7]:
tools = [add, multiply, divide]
tools_by_name = {tool.name: tool for tool in tools}
model_with_tools = model.bind_tools(tools)


In [8]:
from langgraph.graph import add_messages
from langchain.messages import (
    SystemMessage,
    HumanMessage,
    ToolCall,
)
from langchain_core.messages import BaseMessage
from langgraph.func import entrypoint, task

In [9]:
@task
def call_llm(messages: list[BaseMessage]):
    """LLM decides whether to call a tool or not"""
    return model_with_tools.invoke(
        [
            SystemMessage(
                content="You are a helpful assistant tasked with performing arithmetic on a set of inputs."
            )
        ]
        + messages
    )

In [10]:
@task
def call_tool(tool_call: ToolCall):
    """Performs the tool call"""
    tool = tools_by_name[tool_call["name"]]
    return tool.invoke(tool_call)

In [11]:
@entrypoint()
def agent(messages: list[BaseMessage]):
    model_response = call_llm(messages).result()

    while True:
        if not model_response.tool_calls:
            break

        # Execute tools
        tool_result_futures = [
            call_tool(tool_call) for tool_call in model_response.tool_calls
        ]
        tool_results = [fut.result() for fut in tool_result_futures]
        messages = add_messages(messages, [model_response, *tool_results])
        model_response = call_llm(messages).result()

    messages = add_messages(messages, model_response)
    return messages



In [16]:
# Invoke
messages = [HumanMessage(content="Add 3 and 9. and then multiply the result by 2.")]
for chunk in agent.stream(messages, stream_mode="updates"):
    print(chunk)
    print("\n")

{'call_llm': AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gpt-oss:120b', 'created_at': '2026-05-18T13:18:07.200284341Z', 'done': True, 'done_reason': 'stop', 'total_duration': 980152544, 'load_duration': None, 'prompt_eval_count': 262, 'prompt_eval_duration': None, 'eval_count': 70, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:120b', 'model_provider': 'ollama'}, id='lc_run--019e3b3c-95ef-71e0-b73e-0d0e998964d1-0', tool_calls=[{'name': 'add', 'args': {'a': 3, 'b': 9}, 'id': '55bd007b-37b2-4d8e-b98a-50faf1a40e7d', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 262, 'output_tokens': 70, 'total_tokens': 332})}


{'call_tool': ToolMessage(content='12', name='add', id='da941549-f3f6-40ab-8c7f-37e00896c63e', tool_call_id='55bd007b-37b2-4d8e-b98a-50faf1a40e7d')}


{'call_llm': AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gpt-oss:120b', 'created_at': '2026-05-18T13:18:08.396674958Z', 'done'